# 09 — Figures + Tables for the Paper

Produce all paper-ready outputs:

* Figure 1 — country-year exposure trajectories with vintage break
* Figure 2 — caterpillar plot of country random intercepts (M3a)
* Figure 3 — conditional effects of within × education (M5)
* Figure 4 — variance components M0 → M6
* Table 1 — descriptive statistics
* Table 2 — master regression M0 → M6
* Table 3 — Mundlak Wald + robustness summary

All figures saved as `.pdf` to `paper/figures/`; all tables as `.tex` to `paper/tables/`.

In [1]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")  # headless
import matplotlib.pyplot as plt
import pandas as pd

warnings.filterwarnings("ignore")

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT.name != "MLA" and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

FIGURES_DIR = REPO_ROOT / "paper" / "figures"
TABLES_DIR  = REPO_ROOT / "paper" / "tables"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

INTERIM_DIR = REPO_ROOT / "data" / "interim"
ANALYSIS_DIR = REPO_ROOT / "data" / "analysis"

def format_latex_table(tex: str, *, resize: bool = False,
                       font_size: str | None = None,
                       tabcolsep: str | None = None) -> str:
    """Add stable paper layout around a pandas-generated LaTeX table."""
    lines = tex.splitlines()
    tab_start = next(i for i, line in enumerate(lines) if line.startswith(r"\begin{tabular}"))
    tab_end = next(i for i, line in enumerate(lines) if line.startswith(r"\end{tabular}"))

    if not any(line.strip() == r"\centering" for line in lines[:tab_start]):
        lines.insert(1, r"\centering")
        tab_start += 1
        tab_end += 1

    controls = []
    if font_size:
        controls.append(font_size)
    if tabcolsep:
        controls.append(rf"\setlength{{\tabcolsep}}{{{tabcolsep}}}")

    before = lines[:tab_start]
    tabular = lines[tab_start:tab_end + 1]
    after = lines[tab_end + 1:]
    if resize:
        tabular = [r"\resizebox{\linewidth}{!}{%"] + tabular + ["}%"]

    return "\n".join(before + controls + tabular + after) + "\n"

from src.mla.models import (  # noqa: E402
    add_country_year_key, build_trust_composite, fit_3level,
)
from src.mla.plotting import (  # noqa: E402
    caterpillar_country_intercepts,
    conditional_within_x_education,
    country_year_exposure_trajectories,
    variance_components_bar,
)
REPO_ROOT

PosixPath('/Users/karlalucic/Code/coursework/KUL/2sem/MLA')

## 1. Reproduce the analysis frame (same prep as notebooks 06/07/08)

In [2]:
def recode_sentinels(s, lo, hi):
    return s.where(s.between(lo, hi))

df = pd.read_parquet(ANALYSIS_DIR / "analysis.parquet")
df = build_trust_composite(df)
df = add_country_year_key(df)
df["agea"]    = recode_sentinels(df["agea"], 14, 110)
df["gndr"]    = recode_sentinels(df["gndr"], 1, 2)
df["eisced"]  = recode_sentinels(df["eisced"], 0, 7)
df["hinctnta"] = recode_sentinels(df["hinctnta"], 1, 10)
df["mnactic"] = recode_sentinels(df["mnactic"], 1, 9)
df["domicil"] = recode_sentinels(df["domicil"], 1, 5)
df["agea_c"] = df["agea"] - 45
df["agea_c_sq"] = df["agea_c"] ** 2
df["female"] = (df["gndr"] == 2).astype("float64")
for _c in ("essround", "isco08", "year"):
    if str(df[_c].dtype).startswith("Int"):
        df[_c] = df[_c].astype("float64")
df["genai_z"] = (df["genai_i"] - df["genai_i"].mean()) / df["genai_i"].std()
df["eisced_c"] = df["eisced"] - 4
REQUIRED = [
    "trust", "genai_i", "genai_z", "eisced", "eisced_c", "agea_c", "female",
    "mnactic", "domicil", "hinctnta", "gdp_growth", "unemp_rate",
    "hicp_inflation", "exposure_ct", "exposure_ct_within", "exposure_ct_between",
]
df_fit = df.dropna(subset=REQUIRED).copy()
g = df_fit.groupby("cntry", observed=True)["genai_z"].transform("mean")
df_fit["genai_z_gmc"] = df_fit["genai_z"] - g
print(f"analysis sample: {len(df_fit):,}, {df_fit.cntry.nunique()} countries")

analysis sample: 165,969, 30 countries


## 2. Figure 1 — country-year exposure trajectories

In [3]:
cy = pd.read_parquet(INTERIM_DIR / "country_year_exposure.parquet")
fig1 = country_year_exposure_trajectories(cy)
fig1.savefig(FIGURES_DIR / "fig1_exposure_trajectories.pdf", bbox_inches="tight")
fig1.savefig(FIGURES_DIR / "fig1_exposure_trajectories.png", dpi=160, bbox_inches="tight")
plt.close(fig1)
print("wrote figures: fig1_exposure_trajectories.{pdf,png}")

wrote figures: fig1_exposure_trajectories.{pdf,png}


## 3. Figure 2 — caterpillar of country random intercepts (M3a)

In [4]:
M3A_FORMULA = (
    "trust ~ genai_z + C(eisced) + agea_c + agea_c_sq + female "
    "+ C(mnactic) + C(domicil) + hinctnta "
    "+ gdp_growth + unemp_rate + hicp_inflation + C(essround) "
    "+ exposure_ct_within + exposure_ct_between"
)
res3a = fit_3level(M3A_FORMULA, df_fit)
fig2 = caterpillar_country_intercepts(res3a)
fig2.savefig(FIGURES_DIR / "fig2_caterpillar.pdf", bbox_inches="tight")
fig2.savefig(FIGURES_DIR / "fig2_caterpillar.png", dpi=160, bbox_inches="tight")
plt.close(fig2)
print("wrote figures: fig2_caterpillar.{pdf,png}")

wrote figures: fig2_caterpillar.{pdf,png}


/Users/karlalucic/Code/coursework/KUL/2sem/MLA/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


## 4. Figure 3 — conditional within × education (M5)

In [5]:
M5_FORMULA = M3A_FORMULA + " + exposure_ct_within:eisced_c"
res5 = fit_3level(M5_FORMULA, df_fit, re_formula="~1 + genai_z_gmc")
fig3 = conditional_within_x_education(res5)
fig3.savefig(FIGURES_DIR / "fig3_conditional_within_x_education.pdf", bbox_inches="tight")
fig3.savefig(FIGURES_DIR / "fig3_conditional_within_x_education.png", dpi=160, bbox_inches="tight")
plt.close(fig3)
print("wrote figures: fig3_conditional_within_x_education.{pdf,png}")

/Users/karlalucic/Code/coursework/KUL/2sem/MLA/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


wrote figures: fig3_conditional_within_x_education.{pdf,png}


## 5. Figure 4 — variance components M0 → M6

In [6]:
tbl_m0_m6 = pd.read_parquet(INTERIM_DIR / "master_table_m0_m6.parquet")
fig4 = variance_components_bar(tbl_m0_m6)
fig4.savefig(FIGURES_DIR / "fig4_variance_components.pdf", bbox_inches="tight")
fig4.savefig(FIGURES_DIR / "fig4_variance_components.png", dpi=160, bbox_inches="tight")
plt.close(fig4)
print("wrote figures: fig4_variance_components.{pdf,png}")

wrote figures: fig4_variance_components.{pdf,png}


## 6. Table 1 — descriptive statistics

In [ ]:
desc_vars = ["trust", "genai_z", "agea", "female", "hinctnta",
             "gdp_growth", "unemp_rate", "hicp_inflation",
             "exposure_ct", "exposure_ct_within", "exposure_ct_between"]
labels = {
    "trust": "Trust composite (z)",
    "genai_z": "Individual ILO–NASK GenAI exposure (z)",
    "agea": "Age (years)",
    "female": "Female (0/1)",
    "hinctnta": "Household income decile (1–10)",
    "gdp_growth": "Real GDP growth (\\%)",
    "unemp_rate": "Unemployment rate (\\%)",
    "hicp_inflation": "HICP inflation (\\%)",
    "exposure_ct": r"$\bar E_{ct}$ (country-year exposure)",
    "exposure_ct_within": r"$E_{ct} - \bar E_c$ (within)",
    "exposure_ct_between": r"$\bar E_c$ (between)",
}
desc = df_fit[desc_vars].describe().T
desc.index = [labels[v] for v in desc.index]
# Drop the N column: every row is the M3a complete-case sample (N stated in
# the caption), so the column is a constant 165,969 and uninformative.
desc = desc[["mean", "std", "min", "50%", "max"]]
desc.columns = ["Mean", "SD", "Min", "Median", "Max"]
desc_latex = desc.round(2).copy()
desc_latex.index.name = None
tex = desc_latex.to_latex(
    escape=False, column_format=r"p{0.42\linewidth}rrrrr",
    float_format="%.2f",
    caption="Descriptive statistics, M3a analysis sample (N = 165{,}969).",
    label="tab:descriptives",
)
tex = format_latex_table(tex, resize=False, font_size=r"\scriptsize", tabcolsep="3pt")
(TABLES_DIR / "tab1_descriptives.tex").write_text(tex)
print("wrote paper/tables/tab1_descriptives.tex")
desc.round(2)


## 7. Table 2 — master regression M0 → M6

In [ ]:
# Build a compact paper-style table: one column per model, rows for the
# coefficients of interest + variance components.
# Use \makecell[r]{...\\...} (not \newline) for stacked beta/SE within an r-column.

PRETTY_TERM = {
    "genai_z":                          r"Individual GenAI exposure ($z$)",
    "exposure_ct_within":               r"$E_{ct}-\bar E_c$ (within)",
    "exposure_ct_between":              r"$\bar E_c$ (between)",
    "exposure_ct_within:eisced_c":      r"Within $\times$ ISCED",
    "epl_c_centred":                    r"EPL (centred)",
    "exposure_ct_within:epl_c_centred": r"Within $\times$ EPL",
}

tbl = tbl_m0_m6.copy()
rows = []
for coef, pretty in PRETTY_TERM.items():
    row = {"term": pretty}
    for _, r in tbl.iterrows():
        beta = r.get(f"{coef}__beta", float("nan"))
        se = r.get(f"{coef}__se", float("nan"))
        if pd.isna(beta):
            row[r["model"]] = ""
        else:
            z = beta / se if se else float("nan")
            sig = "^{*}" if abs(z) > 1.96 else ""
            row[r["model"]] = f"\\makecell[r]{{${beta:+.3f}{sig}$\\\\({se:.3f})}}"
    rows.append(row)
rows.append({"term": r"$\sigma^2_{v_0}$ (L3)", **{r["model"]: f"{r['sigma_u0_sq']:.3f}" for _, r in tbl.iterrows()}})
rows.append({"term": r"$\sigma^2_{u_0}$ (L2)", **{r["model"]: f"{r['sigma_v0_sq']:.3f}" for _, r in tbl.iterrows()}})
rows.append({"term": r"$\sigma^2_e$  (L1)",   **{r["model"]: f"{r['sigma_e_sq']:.3f}"  for _, r in tbl.iterrows()}})
rows.append({"term": "ICC L3",                  **{r["model"]: f"{r['icc_l3']:.3f}"      for _, r in tbl.iterrows()}})
rows.append({"term": "VPC L2",                  **{r["model"]: f"{r['vpc_l2']:.3f}"      for _, r in tbl.iterrows()}})
rows.append({"term": "N",                       **{r["model"]: f"{int(r['n_obs']):,}"    for _, r in tbl.iterrows()}})
rows.append({"term": "countries (L3)",          **{r["model"]: f"{int(r['n_groups_l3'])}" for _, r in tbl.iterrows()}})
wide = pd.DataFrame(rows).set_index("term")
wide.index.name = None  # avoid the extra blank row pandas inserts
tex2 = wide.to_latex(
    escape=False, column_format="l" + "r" * len(tbl),
    caption=("Three-level multilevel models, M0--M6. Coefficients "
             "(standard errors). $^{*}\\,p<0.05$. M5 is the primary H4 "
             "test; M6 is the OECD/EPL subsample."),
    label="tab:models",
)
tex2 = format_latex_table(tex2, resize=True, font_size=r"\scriptsize", tabcolsep="2pt")
(TABLES_DIR / "tab2_models_m0_m6.tex").write_text(tex2)
print("wrote paper/tables/tab2_models_m0_m6.tex")
wide


## 8. Table 3 — Mundlak Wald + robustness summary

In [ ]:
wald = pd.read_parquet(INTERIM_DIR / "mundlak_wald_tests.parquet")
robust = pd.read_csv(INTERIM_DIR / "robustness_summary.csv")

# Tab 3a: format floats to 3 decimals (otherwise pandas defaults to 6 trailing zeros)
wald_disp = wald.copy()
wald_disp.columns = ["Model", r"$\hat\gamma_W$", r"$\hat\gamma_B$",
                     r"Diff", r"SE(diff)", r"$\chi^2$", "df", r"$p$"]
wald_disp[r"$p$"] = wald_disp[r"$p$"].map(
    lambda p: r"$<0.001$" if 0 < p < 0.0005 else f"{p:.3f}"
)
tex3a = wald_disp.to_latex(
    index=False, escape=False, float_format="%.3f",
    caption=r"Mundlak Wald test of $\gamma_W = \gamma_B$ across the M3a--M5 specifications.",
    label="tab:mundlak",
)
tex3a = format_latex_table(tex3a, resize=False, font_size=r"\scriptsize", tabcolsep="3pt")

# Tab 3b: clean up the analyst-facing intermediate labels into paper-ready prose,
# escape any bare Greek to math-mode, and replace numeric NaN with em-dashes.
# Force a 3-decimal display so trstprl's 1.3826 doesn't sit next to LOO's 0.06,
# and use "<0.001" for tiny p-values that would otherwise round to a misleading 0.000.
LABEL_MAP = {
    "R5 trstprl alone": r"Single item: \texttt{trstprl}",
    "R5 trstlgl alone": r"Single item: \texttt{trstlgl}",
    "R5 stfdem alone":  r"Single item: \texttt{stfdem}",
    "R6 country drop range (γ_B min/max)":
        r"Country drop range ($\hat\gamma_B$ min/max)",
    "R12 drop R10": r"Drop ESS R10 (COVID-disrupted)",
    "R-V vintage-static": r"Vintage-static (2025 throughout)",
    "R4 leave-one-out": r"Leave-one-out cell aggregation",
    "R7 PanelOLS β on exposure_ct (within-only FE)":
        r"Within-only FE benchmark (\texttt{linearmodels.PanelOLS})",
}

def _fmt_cell(x, *, is_pvalue=False):
    """Numeric → 3-decimal; NaN → em-dash; pre-formatted strings (e.g. interval
    ranges) pass through unchanged. P-values that would round to 0.000 print
    as $<0.001$ instead."""
    if isinstance(x, str):
        return x
    try:
        if pd.isna(x):
            return "---"
        v = float(x)
        if is_pvalue and 0 < v < 0.0005:
            return r"$<0.001$"
        return f"{v:.3f}"
    except (TypeError, ValueError):
        return str(x)

robust_disp = robust.copy()
robust_disp["check"] = robust_disp["check"].map(lambda s: LABEL_MAP.get(s, s))
for col in ("gamma_w", "gamma_b"):
    robust_disp[col] = robust_disp[col].map(_fmt_cell)
robust_disp["wald_p"] = robust_disp["wald_p"].map(lambda x: _fmt_cell(x, is_pvalue=True))
robust_disp.columns = ["Check", r"$\hat\gamma_W$", r"$\hat\gamma_B$", r"Wald $p$"]

tex3b = robust_disp.to_latex(
    index=False, escape=False,
    column_format=r"p{0.48\linewidth}rrr",
    caption="Robustness battery for the M3a within--between specification.",
    label="tab:robustness",
)
tex3b = format_latex_table(tex3b, font_size=r"\scriptsize", tabcolsep="4pt")
(TABLES_DIR / "tab3a_mundlak.tex").write_text(tex3a)
(TABLES_DIR / "tab3b_robustness.tex").write_text(tex3b)
print("wrote paper/tables/tab3a_mundlak.tex, tab3b_robustness.tex")
print()
print("Mundlak Wald:")
print(wald.round(4))
print()
print("Robustness:")
print(robust)


## 9. Verify all paper outputs landed

In [10]:
for d in (FIGURES_DIR, TABLES_DIR):
    print(f"--- {d.relative_to(REPO_ROOT)}")
    for p in sorted(d.iterdir()):
        print(f"  {p.name}  ({p.stat().st_size/1e3:.1f} KB)")

--- paper/figures
  .gitkeep  (0.0 KB)
  fig1_exposure_trajectories.pdf  (32.0 KB)
  fig1_exposure_trajectories.png  (147.8 KB)
  fig2_caterpillar.pdf  (19.7 KB)
  fig2_caterpillar.png  (52.8 KB)
  fig3_conditional_within_x_education.pdf  (19.7 KB)
  fig3_conditional_within_x_education.png  (106.8 KB)
  fig4_variance_components.pdf  (14.2 KB)
  fig4_variance_components.png  (34.3 KB)
--- paper/tables
  .gitkeep  (0.0 KB)
  tab1_descriptives.tex  (1.3 KB)
  tab2_models_m0_m6.tex  (1.8 KB)
  tab3a_mundlak.tex  (0.6 KB)
  tab3b_robustness.tex  (0.7 KB)
